# 📦 Supply Chain & Inventory Optimization Dashboard
**Project by:** [Your Name]  
**Tools:** Python, Pandas, Matplotlib  
**Goal:** Analyze inventory data to identify stockout risks, optimize reorder points, and improve supply chain efficiency.

---

## 📌 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (already available in Colab, but listed for clarity)
# !pip install pandas matplotlib seaborn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ All libraries imported successfully!')

## 📌 Step 2: Generate Realistic Sample Dataset

In [ ]:
np.random.seed(42)

# ---- Product Master Data ----
n_products = 50
categories = ['Engine Parts', 'Brake System', 'Electrical', 'Transmission', 'Body Parts']
suppliers  = ['Supplier_A', 'Supplier_B', 'Supplier_C', 'Supplier_D', 'Supplier_E']
warehouses = ['Bangalore_DC', 'Chennai_DC', 'Mumbai_DC']

products = pd.DataFrame({
    'Product_ID'      : [f'SKU-{1000+i}' for i in range(n_products)],
    'Product_Name'    : [f'Part_{chr(65+i%26)}{i}' for i in range(n_products)],
    'Category'        : np.random.choice(categories, n_products),
    'Supplier'        : np.random.choice(suppliers,  n_products),
    'Warehouse'       : np.random.choice(warehouses, n_products),
    'Unit_Cost_INR'   : np.random.randint(500, 15000, n_products),
    'Lead_Time_Days'  : np.random.randint(3, 21, n_products),
    'Safety_Stock'    : np.random.randint(10, 50, n_products),
    'Current_Stock'   : np.random.randint(0, 200, n_products),
    'Max_Stock'       : np.random.randint(200, 500, n_products),
    'Monthly_Demand'  : np.random.randint(20, 150, n_products),
})

# ---- Monthly Sales/Demand Data (12 months) ----
n_records = 500
dates = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')

transactions = pd.DataFrame({
    'Date'        : np.random.choice(dates, n_records),
    'Product_ID'  : np.random.choice(products['Product_ID'], n_records),
    'Quantity_Out': np.random.randint(1, 30, n_records),
    'Quantity_In' : np.random.randint(0, 50, n_records),
    'Order_Status': np.random.choice(['Delivered', 'In Transit', 'Delayed'], n_records,
                                      p=[0.70, 0.20, 0.10]),
})
transactions['Date'] = pd.to_datetime(transactions['Date'])
transactions['Month'] = transactions['Date'].dt.to_period('M')

print(f'✅ Products dataset: {products.shape}')
print(f'✅ Transactions dataset: {transactions.shape}')
print()
print(products.head(5))

## 📌 Step 3: Data Cleaning & Validation

In [ ]:
# Check for missing values
print('=== Missing Values Check ===')
print(products.isnull().sum())
print()

# Check data types
print('=== Data Types ===')
print(products.dtypes)
print()

# Basic statistics
print('=== Inventory Summary Statistics ===')
print(products[['Current_Stock','Monthly_Demand','Unit_Cost_INR','Lead_Time_Days']].describe())

## 📌 Step 4: Inventory Analysis — Stockout Risk & Reorder Point

In [ ]:
# ---- Calculate Key Metrics ----

# Daily demand
products['Daily_Demand'] = products['Monthly_Demand'] / 30

# Reorder Point = (Daily Demand × Lead Time) + Safety Stock
products['Reorder_Point'] = (products['Daily_Demand'] * products['Lead_Time_Days']).round(0) + products['Safety_Stock']

# Days of stock remaining
products['Days_of_Stock'] = (products['Current_Stock'] / products['Daily_Demand']).round(1)

# Stock Status
def stock_status(row):
    if row['Current_Stock'] <= row['Safety_Stock']:
        return 'CRITICAL'
    elif row['Current_Stock'] <= row['Reorder_Point']:
        return 'REORDER NOW'
    elif row['Current_Stock'] >= row['Max_Stock'] * 0.9:
        return 'OVERSTOCK'
    else:
        return 'HEALTHY'

products['Stock_Status'] = products.apply(stock_status, axis=1)

# Inventory Value
products['Inventory_Value_INR'] = products['Current_Stock'] * products['Unit_Cost_INR']

print('=== Stock Status Summary ===')
print(products['Stock_Status'].value_counts())
print()
print('=== Total Inventory Value ===')
print(f"₹ {products['Inventory_Value_INR'].sum():,.0f}")

## 📌 Step 5: ABC Analysis (Inventory Prioritization)

In [ ]:
# ABC Analysis — classify products by revenue/value contribution
# A = Top 70% value | B = Next 20% | C = Bottom 10%

products_sorted = products.sort_values('Inventory_Value_INR', ascending=False).copy()
products_sorted['Cumulative_Value'] = products_sorted['Inventory_Value_INR'].cumsum()
total_value = products_sorted['Inventory_Value_INR'].sum()
products_sorted['Cumulative_Pct'] = products_sorted['Cumulative_Value'] / total_value * 100

def abc_class(pct):
    if pct <= 70: return 'A'
    elif pct <= 90: return 'B'
    else: return 'C'

products_sorted['ABC_Class'] = products_sorted['Cumulative_Pct'].apply(abc_class)
products['ABC_Class'] = products_sorted['ABC_Class']
products['ABC_Class'] = products['ABC_Class'].fillna('C')

print('=== ABC Classification ===')
abc_summary = products.groupby('ABC_Class').agg(
    Products=('Product_ID','count'),
    Total_Value=('Inventory_Value_INR','sum')
).round(0)
abc_summary['Value_%'] = (abc_summary['Total_Value'] / abc_summary['Total_Value'].sum() * 100).round(1)
print(abc_summary)

## 📌 Step 6: Visualizations — Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Supply Chain & Inventory Optimization Dashboard', fontsize=16, fontweight='bold', y=1.01)

colors_status = {'CRITICAL': '#e74c3c', 'REORDER NOW': '#f39c12', 'HEALTHY': '#2ecc71', 'OVERSTOCK': '#3498db'}

# ---- Chart 1: Stock Status Distribution ----
ax1 = axes[0, 0]
status_counts = products['Stock_Status'].value_counts()
bar_colors = [colors_status[s] for s in status_counts.index]
bars = ax1.bar(status_counts.index, status_counts.values, color=bar_colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Stock Status Distribution', fontweight='bold')
ax1.set_xlabel('Status')
ax1.set_ylabel('Number of SKUs')
for bar, val in zip(bars, status_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha='center', fontweight='bold')
ax1.set_facecolor('#f8f9fa')

# ---- Chart 2: Inventory Value by Category ----
ax2 = axes[0, 1]
cat_value = products.groupby('Category')['Inventory_Value_INR'].sum().sort_values(ascending=True)
bars2 = ax2.barh(cat_value.index, cat_value.values / 1e6, color='#2980b9', edgecolor='white')
ax2.set_title('Inventory Value by Category (₹ Millions)', fontweight='bold')
ax2.set_xlabel('Value (₹ Millions)')
for bar, val in zip(bars2, cat_value.values):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'₹{val/1e6:.1f}M', va='center', fontsize=8)
ax2.set_facecolor('#f8f9fa')

# ---- Chart 3: ABC Analysis Pie ----
ax3 = axes[0, 2]
abc_val = products.groupby('ABC_Class')['Inventory_Value_INR'].sum()
abc_colors = ['#e74c3c', '#f39c12', '#95a5a6']
wedges, texts, autotexts = ax3.pie(
    abc_val.values, labels=abc_val.index,
    autopct='%1.1f%%', colors=abc_colors,
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts: t.set_fontsize(10)
ax3.set_title('ABC Analysis — Inventory Value Split', fontweight='bold')

# ---- Chart 4: Monthly Transaction Volume ----
ax4 = axes[1, 0]
monthly = transactions.groupby('Month').agg(
    Qty_Out=('Quantity_Out','sum'),
    Qty_In=('Quantity_In','sum')
).reset_index()
monthly['Month_Label'] = monthly['Month'].astype(str).str[-5:]
x = range(len(monthly))
w = 0.35
ax4.bar([i - w/2 for i in x], monthly['Qty_Out'], width=w, label='Outbound', color='#e74c3c', alpha=0.85)
ax4.bar([i + w/2 for i in x], monthly['Qty_In'],  width=w, label='Inbound',  color='#2ecc71', alpha=0.85)
ax4.set_xticks(list(x))
ax4.set_xticklabels(monthly['Month_Label'], rotation=45, ha='right')
ax4.set_title('Monthly Inbound vs Outbound Volume', fontweight='bold')
ax4.set_ylabel('Quantity')
ax4.legend()
ax4.set_facecolor('#f8f9fa')

# ---- Chart 5: Supplier Lead Time Comparison ----
ax5 = axes[1, 1]
supplier_lt = products.groupby('Supplier')['Lead_Time_Days'].mean().sort_values(ascending=False)
bars5 = ax5.bar(supplier_lt.index, supplier_lt.values, color='#8e44ad', edgecolor='white', alpha=0.85)
ax5.set_title('Avg Lead Time by Supplier (Days)', fontweight='bold')
ax5.set_xlabel('Supplier')
ax5.set_ylabel('Lead Time (Days)')
ax5.axhline(supplier_lt.mean(), color='red', linestyle='--', label=f'Avg: {supplier_lt.mean():.1f} days')
ax5.legend()
for bar, val in zip(bars5, supplier_lt.values):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.1f}d', ha='center', fontsize=9)
ax5.set_facecolor('#f8f9fa')

# ---- Chart 6: Order Status Breakdown ----
ax6 = axes[1, 2]
order_status = transactions['Order_Status'].value_counts()
status_colors = ['#2ecc71', '#3498db', '#e74c3c']
wedges6, texts6, auto6 = ax6.pie(
    order_status.values, labels=order_status.index,
    autopct='%1.1f%%', colors=status_colors,
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax6.set_title('Order Fulfillment Status', fontweight='bold')

plt.tight_layout()
plt.savefig('inventory_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved as inventory_dashboard.png')

## 📌 Step 7: Critical Items — Actionable Alerts

In [ ]:
print('=' * 60)
print('🚨 CRITICAL STOCK ALERT — IMMEDIATE ACTION REQUIRED')
print('=' * 60)
critical = products[products['Stock_Status'] == 'CRITICAL'][[
    'Product_ID','Product_Name','Category','Supplier',
    'Current_Stock','Safety_Stock','Daily_Demand','Days_of_Stock'
]].sort_values('Days_of_Stock')
print(critical.to_string(index=False))

print()
print('=' * 60)
print('⚠️  REORDER REQUIRED — ORDER WITHIN LEAD TIME')
print('=' * 60)
reorder = products[products['Stock_Status'] == 'REORDER NOW'][[
    'Product_ID','Product_Name','Supplier','Current_Stock',
    'Reorder_Point','Lead_Time_Days','Days_of_Stock'
]].sort_values('Days_of_Stock')
print(reorder.head(10).to_string(index=False))

## 📌 Step 8: KPI Summary Report

In [ ]:
total_skus       = len(products)
total_value      = products['Inventory_Value_INR'].sum()
critical_count   = (products['Stock_Status'] == 'CRITICAL').sum()
reorder_count    = (products['Stock_Status'] == 'REORDER NOW').sum()
overstock_count  = (products['Stock_Status'] == 'OVERSTOCK').sum()
healthy_count    = (products['Stock_Status'] == 'HEALTHY').sum()
avg_lead_time    = products['Lead_Time_Days'].mean()
delayed_pct      = (transactions['Order_Status'] == 'Delayed').sum() / len(transactions) * 100
avg_days_stock   = products['Days_of_Stock'].mean()

print('=' * 50)
print('       📊 SUPPLY CHAIN KPI DASHBOARD SUMMARY')
print('=' * 50)
print(f'  Total SKUs Tracked        : {total_skus}')
print(f'  Total Inventory Value     : ₹ {total_value:>12,.0f}')
print(f'  Avg Days of Stock         : {avg_days_stock:.1f} days')
print(f'  Avg Supplier Lead Time    : {avg_lead_time:.1f} days')
print(f'  Order Delay Rate          : {delayed_pct:.1f}%')
print('-' * 50)
print(f'  🟢 Healthy Stock          : {healthy_count} SKUs')
print(f'  🔵 Overstock              : {overstock_count} SKUs')
print(f'  🟡 Reorder Required       : {reorder_count} SKUs')
print(f'  🔴 Critical / Stockout    : {critical_count} SKUs')
print('=' * 50)

## 📌 Step 9: Export to CSV (Power BI Ready)

In [ ]:
# Export clean datasets for Power BI
products.to_csv('inventory_master.csv', index=False)
transactions.to_csv('transactions.csv', index=False)

# Export summary for Power BI dashboard
summary = products.groupby(['Category', 'Stock_Status', 'ABC_Class']).agg(
    SKU_Count=('Product_ID', 'count'),
    Total_Stock=('Current_Stock', 'sum'),
    Total_Value=('Inventory_Value_INR', 'sum'),
    Avg_Lead_Time=('Lead_Time_Days', 'mean')
).reset_index()
summary.to_csv('powerbi_summary.csv', index=False)

print('✅ Files exported for Power BI:')
print('   → inventory_master.csv   (product-level data)')
print('   → transactions.csv       (transaction history)')
print('   → powerbi_summary.csv    (aggregated KPIs)')
print()
print('📊 Power BI Steps:')
print('   1. Open Power BI Desktop')
print('   2. Get Data → CSV → import all 3 files')
print('   3. Link tables via Product_ID')
print('   4. Build visuals using Stock_Status, ABC_Class, Category')

---
## ✅ Project Summary

This project demonstrates end-to-end supply chain & inventory analysis:

| What We Did | Technique Used |
|---|---|
| Generated realistic supply chain data | NumPy + Pandas |
| Cleaned & validated data | Pandas |
| Calculated Reorder Points & Safety Stock | Domain formula |
| Classified inventory risk | Custom logic |
| Prioritized items by value | ABC Analysis |
| Built 6-chart dashboard | Matplotlib |
| Exported for Power BI | CSV export |

**Skills demonstrated:** Data wrangling, supply chain KPIs, inventory optimization, data visualization, Power BI integration readiness.